# Harris County, TX — HAZUS depth-damage and EAD (Phase 1, step 5)

Final link in the local chain. Take `harris_building_depths.parquet`,
assign a HAZUS occupancy archetype to every building from its Overture
`class`/`subtype`, look up dollar damage at the 100-yr and 500-yr depths
from inline HAZUS depth-damage tables, and integrate to **Expected
Annual Damage** (EAD) in dollars.

**Damage model.**
```
replacement_value_$ = footprint_m2 × num_floors × $/sqft(archetype)
damage_$_T          = DDF[archetype](depth_T_ft) × replacement_value_$
EAD_$               = ½ (1/100 − 1/500) (damage_$_100 + damage_$_500)
```

**Why two-point trapezoid only.** CLAUDE.md describes a four-point
(10/50/100/500) GEV-interpolated curve. NFHL only ships 100-yr and
500-yr footprints, so the local prototype runs the trapezoid between
those two and ignores the 10/50-yr tails (which are zero for SFHA
buildings well above the 100-yr WSE anyway). The Phase-2 cloud version
adds the GEV interpolation and a non-zero lower tail.

**Inputs:**
- `data/raw/harris_buildings.parquet` (for footprint geometry → area)
- `data/raw/harris_building_depths.parquet` (from notebook 04)

**Output:** `data/raw/harris_building_ead.parquet`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import folium


In [ ]:
FT_PER_M = 1 / 0.3048
SQFT_PER_M2 = 10.7639
UTM_CRS = "EPSG:32615"

RP_LOW, RP_HIGH = 100, 500  # return periods (years) we have depths for

REPO_ROOT = Path.cwd().resolve().parents[1]
RAW_DIR = REPO_ROOT / "data" / "raw"
BUILDINGS_PATH = RAW_DIR / "harris_buildings.parquet"
DEPTHS_PATH = RAW_DIR / "harris_building_depths.parquet"
OUT_PATH = RAW_DIR / "harris_building_ead.parquet"
OUT_PATH


## Overture → HAZUS archetype mapping

Mapping is per CLAUDE.md §Methodology Component 3. Anything unmatched
falls back to RES1 (single-family residential) — by far the dominant
Harris County class and the safest default. Replacement-value $/sqft is
the rounded HAZUS 6.0 default for Texas, in 2023 dollars.


In [ ]:
ARCHETYPE_RULES: list[tuple[str | None, str | None, str]] = [
    # (overture class, overture subtype substring, hazus archetype)
    ("residential", "multi", "RES3"),
    ("residential", None, "RES1"),
    ("commercial", "retail", "COM1"),
    ("commercial", "office", "COM4"),
    ("commercial", None, "COM1"),
    ("industrial", None, "IND1"),
    ("education", None, "EDU1"),
    ("medical", None, "COM6"),
]

REPLACEMENT_USD_PER_SQFT = {
    "RES1": 110.0, "RES3": 130.0,
    "COM1": 95.0,  "COM4": 145.0, "COM6": 280.0,
    "IND1": 90.0,  "EDU1": 150.0,
}


def assign_archetype(klass: str | None, subtype: str | None) -> str:
    k = klass.lower() if isinstance(klass, str) else ""
    s = subtype.lower() if isinstance(subtype, str) else ""
    for rule_class, rule_sub, arche in ARCHETYPE_RULES:
        if rule_class and rule_class not in k:
            continue
        if rule_sub and rule_sub not in s:
            continue
        return arche
    return "RES1"


## HAZUS depth-damage functions

Inline tables of damage fraction (0–1) vs. depth-above-floor in **feet**
for each archetype. Sourced from HAZUS Technical Manual Inventory 6.0,
Appendix A (FIA average-house curves where HAZUS exposes them, otherwise
the USACE generic counterpart). We linearly interpolate between table
rows and clamp to the endpoints.

These curves are coarse-grained on purpose — the prototype's job is to
exercise the EAD integration, not to publish damage fractions. Phase 2
swaps these for the full HAZUS DDF library loaded from `config/`.


In [ ]:
DDF_TABLES: dict[str, list[tuple[float, float]]] = {
    "RES1": [(-2, 0.00), (0, 0.09), (1, 0.14), (3, 0.27), (6, 0.40), (10, 0.50), (16, 0.60)],
    "RES3": [(-2, 0.00), (0, 0.05), (1, 0.10), (3, 0.21), (6, 0.34), (10, 0.46), (16, 0.55)],
    "COM1": [(-2, 0.00), (0, 0.02), (1, 0.09), (3, 0.18), (6, 0.31), (10, 0.43), (16, 0.50)],
    "COM4": [(-2, 0.00), (0, 0.01), (1, 0.07), (3, 0.16), (6, 0.27), (10, 0.40), (16, 0.48)],
    "COM6": [(-2, 0.00), (0, 0.05), (1, 0.13), (3, 0.25), (6, 0.40), (10, 0.55), (16, 0.65)],
    "IND1": [(-2, 0.00), (0, 0.02), (1, 0.06), (3, 0.13), (6, 0.22), (10, 0.32), (16, 0.40)],
    "EDU1": [(-2, 0.00), (0, 0.04), (1, 0.10), (3, 0.20), (6, 0.32), (10, 0.44), (16, 0.52)],
}


def damage_fraction(archetype: pd.Series, depth_ft: np.ndarray) -> np.ndarray:
    out = np.zeros_like(depth_ft, dtype="float64")
    for code, table in DDF_TABLES.items():
        mask = (archetype == code).to_numpy()
        if not mask.any():
            continue
        xs = np.array([t[0] for t in table])
        ys = np.array([t[1] for t in table])
        out[mask] = np.interp(depth_ft[mask], xs, ys, left=ys[0], right=ys[-1])
    return out


## Load inputs and compute footprint area

Footprint area is needed for replacement value; we project to UTM 15N
and read `geometry.area`. `num_floors` defaults to 1 where Overture
didn't carry a value (about 50% of Harris footprints).


In [ ]:
depths = pd.read_parquet(DEPTHS_PATH)
buildings = gpd.read_parquet(BUILDINGS_PATH).set_crs("EPSG:4326", allow_override=True)
areas = (
    buildings[["id", "geometry"]]
    .to_crs(UTM_CRS)
    .assign(area_m2=lambda d: d.geometry.area)
    [["id", "area_m2"]]
)
df = depths.merge(areas, on="id", how="left")
df["num_floors"] = pd.to_numeric(df["num_floors"], errors="coerce").fillna(1).clip(lower=1)
len(df), df[["area_m2", "num_floors"]].describe()


## Score every building

Replacement value, damage fraction at each return period, dollar damage,
EAD. Vectorized over the full ~1.5 M rows; the per-archetype DDF lookup
is a single `np.interp` per archetype.


In [ ]:
df["archetype"] = [assign_archetype(c, s) for c, s in zip(df["class"], df["subtype"])]
df["replacement_usd"] = (
    df["area_m2"].fillna(0) * SQFT_PER_M2
    * df["num_floors"]
    * df["archetype"].map(REPLACEMENT_USD_PER_SQFT)
)

depth_100_ft = (df["depth_100_m"].astype("float64") * FT_PER_M).to_numpy()
depth_500_ft = (df["depth_500_m"].astype("float64") * FT_PER_M).to_numpy()
df["damage_frac_100"] = damage_fraction(df["archetype"], depth_100_ft)
df["damage_frac_500"] = damage_fraction(df["archetype"], depth_500_ft)
df["damage_usd_100"] = df["damage_frac_100"] * df["replacement_usd"]
df["damage_usd_500"] = df["damage_frac_500"] * df["replacement_usd"]

# trapezoid in probability space between RP_LOW and RP_HIGH
dp = (1 / RP_LOW) - (1 / RP_HIGH)
df["ead_usd"] = 0.5 * dp * (df["damage_usd_100"] + df["damage_usd_500"])
df[["damage_usd_100", "damage_usd_500", "ead_usd"]].describe()


In [ ]:
summary = pd.DataFrame({
    "buildings": [len(df)],
    "in_sfha": [int(df["in_sfha"].sum())],
    "with_damage_100": [int((df["damage_usd_100"] > 0).sum())],
    "total_replacement_$B": [df["replacement_usd"].sum() / 1e9],
    "total_damage_100_$B": [df["damage_usd_100"].sum() / 1e9],
    "total_damage_500_$B": [df["damage_usd_500"].sum() / 1e9],
    "total_ead_$M": [df["ead_usd"].sum() / 1e6],
})
summary.T


In [ ]:
df.groupby("archetype").agg(
    n=("id", "size"),
    in_sfha=("in_sfha", "sum"),
    ead_total_usd=("ead_usd", "sum"),
    ead_mean_usd=("ead_usd", "mean"),
).sort_values("ead_total_usd", ascending=False)


In [ ]:
out_cols = [
    "id", "class", "subtype", "archetype", "FLD_ZONE", "in_sfha",
    "area_m2", "num_floors", "elev_m", "bfe_m",
    "depth_100_m", "depth_500_m",
    "replacement_usd", "damage_frac_100", "damage_frac_500",
    "damage_usd_100", "damage_usd_500", "ead_usd",
]
df[out_cols].to_parquet(OUT_PATH, compression="zstd")
print(f"wrote {OUT_PATH} — {len(df):,} rows")


## Eyeball check: top-EAD buildings on a folium map

Plotting all 1.5 M centroids would crash the browser. Show the top 2 000
by EAD, sized by EAD quintile and red-shifted. Anything that lights up
outside the SFHA polygons drawn in notebook 02 deserves investigation —
it usually means an Overture footprint with an outlier elevation
(elevation sampling missed and clipped to mosaic edge).


In [ ]:
top = df.nlargest(2000, "ead_usd").merge(
    buildings[["id", "geometry"]], on="id", how="left"
)
top = gpd.GeoDataFrame(top, geometry="geometry", crs="EPSG:4326")
top["lon"] = top.geometry.centroid.x
top["lat"] = top.geometry.centroid.y

qs = top["ead_usd"].quantile([0.2, 0.4, 0.6, 0.8]).tolist()

def color_for(v: float) -> str:
    if v >= qs[3]: return "#67000d"
    if v >= qs[2]: return "#a50f15"
    if v >= qs[1]: return "#cb181d"
    if v >= qs[0]: return "#ef3b2c"
    return "#fc9272"

m = folium.Map(location=[top["lat"].mean(), top["lon"].mean()], zoom_start=10, tiles="cartodbpositron")
for _, r in top.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3,
        color=color_for(r["ead_usd"]),
        weight=0,
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{r['archetype']} | {r['FLD_ZONE']} | EAD ${r['ead_usd']:,.0f}/yr",
    ).add_to(m)
m


## Next (Phase 2 — cloud)

1. Lift the chain to GCS-backed parquet + Sedona on Dataproc Serverless;
   replace the in-memory rasterio sample with `RS_Value` over a tiled
   3DEP mosaic.
2. Replace the two-point trapezoid with the four-point GEV-interpolated
   curve described in CLAUDE.md (10/50/100/500-yr depths).
3. Move HAZUS DDF tables to `config/archetypes.yaml` and load through a
   typed dataclass — same shape as here, just sourced from config.
4. Add CDC SVI overlay → `human_risk = ead_usd × svi_overall`.
5. Train the XGBoost susceptibility model on Global Flood Database labels;
   surface SHAP-attributed top drivers per building.
